# Phase 7: SQL Analysis for EDA Business Questions

## Purpose

This notebook loads the final Project 2 EDA dataset into a SQLite database and uses SQL queries to answer business analysis questions.

Project 1 focused on data cleaning and database setup.

Project 2 is different because SQL is used for Exploratory Data Analysis, business insights, and decision-making.

Main question:

What business insights can be discovered from the retail orders dataset using SQL?

In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
df = pd.read_csv("../data/processed/ecommerce_orders_project2_final_eda_dataset.csv")

df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,...,CouponCode,ReferralSource,TotalPrice,HasCoupon,TotalPrice_Outlier_IQR,Year,Month,MonthName,YearMonth,RiskStatus
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,...,SAVE10,Instagram,2853.10,Coupon Used,Normal,2023,1,January,2023-01,Normal Order
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,...,SAVE10,Referral,302.70,Coupon Used,Normal,2024,8,August,2024-08,Normal Order
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,...,FREESHIP,Email,2753.40,Coupon Used,Normal,2024,2,February,2024-02,Risk Order
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,...,SAVE10,Facebook,273.19,Coupon Used,Normal,2023,10,October,2023-10,Risk Order
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,...,SAVE10,Email,2504.04,Coupon Used,Normal,2025,5,May,2025-05,Normal Order


In [3]:
os.makedirs("../data/database", exist_ok=True)

db_path = "../data/database/ecommerce_orders_project2.db"

conn = sqlite3.connect(db_path)

df.to_sql("orders", conn, if_exists="replace", index=False)

print("Database created and dataset loaded successfully.")

Database created and dataset loaded successfully.


In [4]:
#check table structure
query = """
PRAGMA table_info(orders);
"""

pd.read_sql_query(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TEXT,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


In [5]:
#Query 1: Total business overview
query = """
SELECT 
    COUNT(DISTINCT OrderID) AS TotalOrders,
    COUNT(DISTINCT CustomerID) AS TotalCustomers,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders;
"""

business_overview = pd.read_sql_query(query, conn)
business_overview

,TotalOrders,TotalCustomers,TotalRevenue,AverageOrderValue
0,1200,1189,1264761.96,1053.97


In [6]:
#Query 2: Revenue by product
query = """
SELECT 
    Product,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    SUM(Quantity) AS TotalQuantitySold,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC;
"""

product_sql_summary = pd.read_sql_query(query, conn)
product_sql_summary

,Product,TotalOrders,TotalQuantitySold,TotalRevenue,AverageOrderValue
0,Chair,178,562,195620.11,1098.99
1,Printer,181,542,195612.61,1080.73
2,Laptop,173,535,192126.56,1110.56
3,Tablet,179,497,186568.95,1042.28
4,Monitor,163,480,175651.41,1077.62
5,Desk,170,508,167459.93,985.06
6,Phone,156,411,151722.39,972.58


In [7]:
#Query 3: Monthly revenue trend
query = """
SELECT 
    YearMonth,
    COUNT(DISTINCT OrderID) AS MonthlyOrders,
    ROUND(SUM(TotalPrice), 2) AS MonthlyRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY YearMonth
ORDER BY YearMonth;
"""

monthly_sql_summary = pd.read_sql_query(query, conn)
monthly_sql_summary

,YearMonth,MonthlyOrders,MonthlyRevenue,AverageOrderValue
0,2023-01,47,56685.75,1206.08
1,2023-02,37,40117.66,1084.26
2,2023-03,43,48609.37,1130.45
3,2023-04,31,27751.71,895.22
4,2023-05,49,63836.84,1302.79
5,2023-06,45,49500.19,1100.00
6,2023-07,44,42820.66,973.20
7,2023-08,51,54352.14,1065.73
8,2023-09,29,29526.67,1018.16
9,2023-10,47,52607.85,1119.32


In [8]:
#Query 4: Payment method analysis
query = """
SELECT 
    PaymentMethod,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC;
"""

payment_sql_summary = pd.read_sql_query(query, conn)
payment_sql_summary

,PaymentMethod,TotalOrders,TotalRevenue,AverageOrderValue
0,Credit Card,234,263847.63,1127.55
1,Online,258,262442.94,1017.22
2,Cash,246,259786.29,1056.04
3,Gift Card,230,246323.92,1070.97
4,Debit Card,232,232361.18,1001.56


In [9]:
#Query 5: Coupon vs no coupon analysis
query = """
SELECT 
    HasCoupon,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY HasCoupon
ORDER BY AverageOrderValue DESC;
"""

coupon_sql_summary = pd.read_sql_query(query, conn)
coupon_sql_summary

,HasCoupon,TotalOrders,TotalRevenue,AverageOrderValue
0,Coupon Used,891,942360.55,1057.64
1,No Coupon,309,322401.41,1043.37


In [10]:
#Query 6: Referral source performance
query = """
SELECT 
    ReferralSource,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY ReferralSource
ORDER BY TotalRevenue DESC;
"""

referral_sql_summary = pd.read_sql_query(query, conn)
referral_sql_summary

,ReferralSource,TotalOrders,TotalRevenue,AverageOrderValue
0,Instagram,259,275285.45,1062.88
1,Email,250,261808.55,1047.23
2,Google,241,250441.48,1039.18
3,Facebook,228,250410.90,1098.29
4,Referral,222,226815.58,1021.69


In [11]:
#Query 7: Order status analysis
query = """
SELECT 
    OrderStatus,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
    ROUND(AVG(TotalPrice), 2) AS AverageOrderValue
FROM orders
GROUP BY OrderStatus
ORDER BY TotalOrders DESC;
"""

order_status_sql_summary = pd.read_sql_query(query, conn)
order_status_sql_summary

,OrderStatus,TotalOrders,TotalRevenue,AverageOrderValue
0,Cancelled,250,276396.21,1105.58
1,Returned,247,243277.70,984.93
2,Pending,237,256328.15,1081.55
3,Shipped,235,246159.58,1047.49
4,Delivered,231,242600.32,1050.22


In [12]:
#Query 8: Risk orders by product
query = """
SELECT 
    Product,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    SUM(CASE WHEN RiskStatus = 'Risk Order' THEN 1 ELSE 0 END) AS RiskOrders,
    ROUND(
        SUM(CASE WHEN RiskStatus = 'Risk Order' THEN 1 ELSE 0 END) * 100.0 
        / COUNT(DISTINCT OrderID), 
        2
    ) AS RiskOrderRate
FROM orders
GROUP BY Product
ORDER BY RiskOrderRate DESC;
"""

risk_product_sql_summary = pd.read_sql_query(query, conn)
risk_product_sql_summary

,Product,TotalOrders,RiskOrders,RiskOrderRate
0,Monitor,163,71,43.56
1,Tablet,179,77,43.02
2,Laptop,173,74,42.77
3,Chair,178,73,41.01
4,Printer,181,73,40.33
5,Phone,156,62,39.74
6,Desk,170,67,39.41


In [13]:
#Query 9: Risk orders by payment method
query = """
SELECT 
    PaymentMethod,
    COUNT(DISTINCT OrderID) AS TotalOrders,
    SUM(CASE WHEN RiskStatus = 'Risk Order' THEN 1 ELSE 0 END) AS RiskOrders,
    ROUND(
        SUM(CASE WHEN RiskStatus = 'Risk Order' THEN 1 ELSE 0 END) * 100.0 
        / COUNT(DISTINCT OrderID), 
        2
    ) AS RiskOrderRate
FROM orders
GROUP BY PaymentMethod
ORDER BY RiskOrderRate DESC;
"""

risk_payment_sql_summary = pd.read_sql_query(query, conn)
risk_payment_sql_summary

,PaymentMethod,TotalOrders,RiskOrders,RiskOrderRate
0,Gift Card,230,102,44.35
1,Credit Card,234,103,44.02
2,Cash,246,106,43.09
3,Debit Card,232,95,40.95
4,Online,258,91,35.27


In [14]:
#Query 10: High-value outlier orders
query = """
SELECT 
    OrderID,
    Date,
    CustomerID,
    Product,
    Quantity,
    UnitPrice,
    TotalPrice,
    PaymentMethod,
    OrderStatus,
    ReferralSource,
    HasCoupon
FROM orders
WHERE TotalPrice_Outlier_IQR = 'Outlier'
ORDER BY TotalPrice DESC;
"""

outlier_orders_sql_summary = pd.read_sql_query(query, conn)
outlier_orders_sql_summary

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,TotalPrice,PaymentMethod,OrderStatus,ReferralSource,HasCoupon
0,ORD200789,2023-08-17,C57276,Tablet,5,691.28,3456.40,Online,Delivered,Email,Coupon Used
1,ORD201122,2023-06-07,C38840,Monitor,5,678.19,3390.95,Online,Returned,Facebook,No Coupon
2,ORD200632,2023-05-02,C67260,Laptop,5,678.16,3390.80,Gift Card,Delivered,Facebook,Coupon Used
3,ORD200469,2023-11-26,C13877,Chair,5,676.98,3384.90,Cash,Cancelled,Facebook,No Coupon
4,ORD200328,2023-02-28,C18404,Tablet,5,674.04,3370.20,Online,Cancelled,Google,Coupon Used
5,ORD200107,2023-03-27,C16775,Printer,5,670.75,3353.75,Gift Card,Shipped,Instagram,Coupon Used
6,ORD200326,2024-07-01,C65986,Laptop,5,670.48,3352.40,Gift Card,Returned,Facebook,Coupon Used
7,ORD201065,2023-10-30,C47778,Printer,5,666.80,3334.00,Debit Card,Delivered,Referral,Coupon Used


In [15]:
#Save SQL result summaries
business_overview.to_csv("../reports/sql_business_overview.csv", index=False)
product_sql_summary.to_csv("../reports/sql_product_summary.csv", index=False)
monthly_sql_summary.to_csv("../reports/sql_monthly_summary.csv", index=False)
payment_sql_summary.to_csv("../reports/sql_payment_summary.csv", index=False)
coupon_sql_summary.to_csv("../reports/sql_coupon_summary.csv", index=False)
referral_sql_summary.to_csv("../reports/sql_referral_summary.csv", index=False)
order_status_sql_summary.to_csv("../reports/sql_order_status_summary.csv", index=False)
risk_product_sql_summary.to_csv("../reports/sql_risk_product_summary.csv", index=False)
risk_payment_sql_summary.to_csv("../reports/sql_risk_payment_summary.csv", index=False)
outlier_orders_sql_summary.to_csv("../reports/sql_outlier_orders_summary.csv", index=False)

print("SQL result summaries saved successfully.")

SQL result summaries saved successfully.


In [16]:
conn.close()

print("Database connection closed.")

Database connection closed.


## SQL Analysis Summary

This phase used SQLite to answer business-focused EDA questions from the final Project 2 dataset.

Project 1 used SQL mainly for database setup and validation.  
Project 2 used SQL differently, focusing on business insights and decision-making.

### Key SQL Findings

- Total Orders: 1,200
- Total Customers: 1,189
- Total Revenue: 1,264,761.96
- Average Order Value: 1,053.97
- Top Product by Revenue: Chair
- Top Payment Method by Revenue: Credit Card
- Top Referral Source by Revenue: Instagram
- Highest Risk Product: Monitor
- Highest Risk Payment Method: Gift Card
- High-value Outlier Orders: 8

### Business Meaning

SQL analysis confirmed the main EDA findings from Python.

The queries helped identify top-performing products, strong referral channels, coupon behavior, order status patterns, risky orders, and high-value outlier orders.

### Conclusion

The SQL phase supports the Project 2 goal by turning the retail orders dataset into business questions, measurable insights, and recruiter-ready analysis.